In [ ]:
import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import ollama

import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("./demo4.csv")


EMBEDDING_MODEL = "BAAI/bge-m3"
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
embeddings = np.array(embedding_model.encode(df["text"].tolist(), normalize_embeddings=True))

# Store in FAISS (cosine similarity)
d, n = embeddings.shape
index = faiss.IndexFlatIP(n)
faiss.normalize_L2(embeddings)
index.add(embeddings)


def retrieve_similar(query, top_k=3):
    query_embedding = embedding_model.encode([query], normalize_embeddings=True)
    _, indices = index.search(query_embedding, top_k)
    return [(df.iloc[i]["title"], df.iloc[i]["content"]) for i in indices[0]]

In [14]:
def generate_answer(query):
    retrieved_docs = retrieve_similar(query)

    context = "\n".join([f"{title}: {content}" for title, content in retrieved_docs])

    # print(context)
    print('retrieved_context: \n: ', context)
    print('\n\n\ --- \n\n')
    
    prompt = f"Answer the question based on the following context:\n{context}\n\nQuestion: {query}\nAnswer:"
    response = ollama.chat(model='hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF', messages=[{"role": "user", "content": prompt}])
    return response["message"]["content"]

In [49]:
query = "Tôi muốn biết về ngành Công nghệ thông tin của trường?"
print(generate_answer(query))

retrieved_context: 
:  Ngành Công nghệ Thông tin (AUN-QA): Trang bị kiến thức về Công nghệ thông tin (Công nghệ phần mềm, Mạng máy tính, Hệ thống thông tin). Cấu trúc dữ liệu và giải thuật, Phương pháp lập trình hướng đối tượng, Công nghệ Phần mềm, Mạng máy tính, Cơ sở dữ liệu, Kiểm thử phần mềm và các kiến thức chuyên sâu khác cần thiết cho người kỹ sư công nghệ thông tin.
Cơ hội việc làm: Kỹ sư thiết kế, phát triển phần mềm, lập trình thiết kế web, giải pháp mạng, hệ thống thông tin, quản trị dữ liệu, nghiên cứu và giảng dạy CNTT.
Ngành Công nghệ truyền thông (Ngành mới tuyển sinh 2025): Ngành Công nghệ truyền thông kết hợp giữ Công nghệ thông tin và truyền thông, với mục tiêu quản lý, tạo ra và phân phối thông tin qua các phương tiện truyền thông hiện đại. Các chuyên ngành trong lĩnh vực này bao gồm truyền thông kỹ thuật số, mạng máy tính, phát triển phần mềm, truyền hình và marketing trực tuyến.
Cơ hội việc làm: Chuyên viên phân tích dữ liệu truyền thông, quản trị mạng, phát triển 

In [16]:
query = "Thông tin tuyển sinh đại học"
print(generate_answer(query))

retrieved_context: 
:  THÔNG TIN VỀ TUYỂN SINH HỆ ĐẠI HỌC CHÍNH QUY: Tuyển thẳng và ưu tiên xét tuyển: Nhận hồ sơ đăng ký từ ngày 01/4 – 30/5/2025 tại http://xettuyen.hcmute.edu.vn
+ Tuyển thẳng theo quy chế của Bộ GD&ĐT.
+ Ưu tiên xét tuyển theo Đề án tuyển sinh của trường (Xét tuyển thí sinh các Trường THPT có ký kết hợp tác)
* Trường Tổ chức thi các môn năng khiếu để xét tuyển vào 4 ngành: Thiết kế thời trang; Thiết kế đồ họa; Kiến trúc; Kiến trúc nội thất. Nhận hồ sơ đăng ký thi môn năng khiếu từ ngày 01/4 – 30/5/2025 tại http://xettuyen.hcmute.edu.vn
Để tăng khả năng trúng tuyển, thí sinh được khuyến nghị đăng ký nhiều phương thức và nhiều nguyện vọng (các nguyện vọng được xét theo thứ tự ưu tiên; nguyện vọng 1 là ưu tiên cao nhất).
Kinh nghiệm qua các năm: mỗi thí sinh đăng ký từ 5 - 7 nguyện vọng, trong đó nguyện vọng từ 1 - 3 nên chọn ĐH SPKT TP. HCM; mỗi mã ngành chỉ đăng ký một tổ hợp có điểm cao nhất. 
Thông tin chi tiết xem tại website: http://t

In [17]:
query = "Các ngành đào tạo mới mở năm 2025 trường đh sư phạm kỹ thuật"
print(generate_answer(query))

retrieved_context: 
:  Các ngành đào tạo năm 2025 học tại trường ĐH SPKT TP. HCM: ```markdown
| TT | Tên ngành đào tạo \n Cấp học bổng học kỳ 1 năm học đầu tiên: bằng 50% học phí cho nữ học 6 ngành kỹ thuật (*) | Chương trình Đào tạo bằng tiếng Việt | Chương trình Đào tạo bằng tiếng Anh | Chương trình Việt - Nhật | Tổ hợp môn xét tuyển dự kiến (in đậm là môn chính nhân hệ số 2) |
|----|--------------------|------------------------------------------------------------------------------------------------|--------------------------------------|--------------------------------------|--------------------------|----------------------------------------------------------------------------------------|
| 1  | Công nghệ Kỹ thuật điện, điện tử | 7510301V | 7510301A | | | **Toán** – Lý – Hóa; **Toán** – Lý – Anh; **Toán** – Văn – Anh; **Toán** – Anh – Công nghệ Công nghiệp; **Toán** – Văn – Lý. |
| 2  | Công nghệ Kỹ thuật điện tử - viễn thông | 7510302V | 7510302A | 7510302N | | |
| 

In [18]:
query = "Trường Đại học Sư phạm Kỹ thuật TP. Hồ Chí Minh (HCMUTE) là trường Đại học công lập hay trường tư?"
print(generate_answer(query))

retrieved_context: 
:  Phần giới thiệu: Trường Đại học Sư phạm Kỹ thuật TP. Hồ Chí Minh là trường Đại học công lập trực thuộc Bộ Giáo dục và Đào tạo, nằm ở TP. Thủ Đức cửa ngõ phía bắc của TP. Hồ Chí Minh, ngay cạnh xa lộ Hà Nội đi các tỉnh miền Đông, miền Trung và miền Bắc. Trường cách trung tâm thành phố khoảng 12 km, phương tiện đi lại, giao thông rất thuận tiện…
10 lý do để bạn nên theo học tại HCMUTE: Trường Công lập với bề dày lịch sử trên 60 năm, thương hiệu hàng đầu phía Nam. 
Tỷ lệ có việc làm đúng chuyên ngành đào tạo rất cao, trên 90%. Được các tập đoàn, doanh nghiệp hàng đầu ưu tiên tuyển dụng.
Chương trình đào tạo đạt chuẩn quốc tế và khu vực, các chương trình liên kết quốc tế, đáp ứng xu thế hội nhập và đào tạo công dân toàn cầu.
Đội ngũ giảng viên được đào tạo bài bản, trình độ cao, nhiều kinh nghiệm thực tiễn, tận tâm. CBVC phục vụ chuyên nghiệp, chu đáo.
Phòng học, phòng Lab/thực tập tiên tiến, đầy đủ, đa dạng; 100% được trang bị máy lạnh.
Đầy ắp các sân chơi học thuật

In [19]:
# df

In [20]:
query = "Trường Đại học Sư phạm Kỹ thuật TP. Hồ Chí Minh (HCMUTE) là trường Đại học công lập hay trường tư?"
print(generate_answer(query))

retrieved_context: 
:  Phần giới thiệu: Trường Đại học Sư phạm Kỹ thuật TP. Hồ Chí Minh là trường Đại học công lập trực thuộc Bộ Giáo dục và Đào tạo, nằm ở TP. Thủ Đức cửa ngõ phía bắc của TP. Hồ Chí Minh, ngay cạnh xa lộ Hà Nội đi các tỉnh miền Đông, miền Trung và miền Bắc. Trường cách trung tâm thành phố khoảng 12 km, phương tiện đi lại, giao thông rất thuận tiện…
10 lý do để bạn nên theo học tại HCMUTE: Trường Công lập với bề dày lịch sử trên 60 năm, thương hiệu hàng đầu phía Nam. 
Tỷ lệ có việc làm đúng chuyên ngành đào tạo rất cao, trên 90%. Được các tập đoàn, doanh nghiệp hàng đầu ưu tiên tuyển dụng.
Chương trình đào tạo đạt chuẩn quốc tế và khu vực, các chương trình liên kết quốc tế, đáp ứng xu thế hội nhập và đào tạo công dân toàn cầu.
Đội ngũ giảng viên được đào tạo bài bản, trình độ cao, nhiều kinh nghiệm thực tiễn, tận tâm. CBVC phục vụ chuyên nghiệp, chu đáo.
Phòng học, phòng Lab/thực tập tiên tiến, đầy đủ, đa dạng; 100% được trang bị máy lạnh.
Đầy ắp các sân chơi học thuật

In [21]:
query = "Vị trí Trường Đại học Sư phạm Kỹ thuật TP. Hồ Chí Minh (HCMUTE) thuận lợi như thế nào cho sinh viên?"
print(generate_answer(query))

retrieved_context: 
:  10 lý do để bạn nên theo học tại HCMUTE: Trường Công lập với bề dày lịch sử trên 60 năm, thương hiệu hàng đầu phía Nam. 
Tỷ lệ có việc làm đúng chuyên ngành đào tạo rất cao, trên 90%. Được các tập đoàn, doanh nghiệp hàng đầu ưu tiên tuyển dụng.
Chương trình đào tạo đạt chuẩn quốc tế và khu vực, các chương trình liên kết quốc tế, đáp ứng xu thế hội nhập và đào tạo công dân toàn cầu.
Đội ngũ giảng viên được đào tạo bài bản, trình độ cao, nhiều kinh nghiệm thực tiễn, tận tâm. CBVC phục vụ chuyên nghiệp, chu đáo.
Phòng học, phòng Lab/thực tập tiên tiến, đầy đủ, đa dạng; 100% được trang bị máy lạnh.
Đầy ắp các sân chơi học thuật, câu lạc bộ nghiên cứu khoa học, sáng tạo - khởi nghiệp.
Các hoạt động văn thể mỹ đa dạng, hấp dẫn giúp sinh viên phát triển toàn diện.
Với triết lý giáo dục Nhân Bản, Trường dành quỹ học bổng lớn hỗ trợ sinh viên. Không để sinh viên phải bỏ học vì không có tiền đóng học phí.
Khuôn viên Trường trên 17 hecta - xanh - sạch - đẹp; là nơi lý tưởng

In [22]:
query = "Ưu thế của trường Đại học Sư phạm Kỹ thuật TP. Hồ Chí Minh là gì?"
print(generate_answer(query))

retrieved_context: 
:  Ưu thế của trường Đại học Sư phạm Kỹ thuật TP. Hồ Chí Minh: Đặc điểm nổi bật trong quá trình đào tạo của trường là: SV tốt nghiệp vừa có kiến thức lý thuyết chuyên sâu đồng thời có tay nghề khá vững (vì trong 4 hoặc 4,5 năm đào tạo, SV có nhiều thời gian học tại phòng thí nghiệm, xưởng thực tập). SV SPKT vừa làm được vừa thuyết trình được vì không những giỏi về chuyên môn mà còn vững vàng về kỹ năng sư phạm. Ưu thế của SV ĐH SPKT Tp. Hồ Chí Minh trong tìm việc là ứng xử, thao diễn các tình huống kỹ thuật công nghệ nhanh & thành thạo. Mức độ hài lòng của các nhà tuyển dụng với SV tốt nghiệp từ ĐH SPKT Tp. Hồ Chí Minh rất cao.
Phần giới thiệu: Trường Đại học Sư phạm Kỹ thuật TP. Hồ Chí Minh là trường Đại học công lập trực thuộc Bộ Giáo dục và Đào tạo, nằm ở TP. Thủ Đức cửa ngõ phía bắc của TP. Hồ Chí Minh, ngay cạnh xa lộ Hà Nội đi các tỉnh miền Đông, miền Trung và miền Bắc. Trường cách trung tâm thành phố khoảng 12 km, phương tiện đi lại, giao thông rất thuận tiện

In [23]:
query = "Hãy nêu cho em biết 1 vài lý do để em mạnh dạn chọn HCMUTE để học đại học?"
print(generate_answer(query))

retrieved_context: 
:  10 lý do để bạn nên theo học tại HCMUTE: Trường Công lập với bề dày lịch sử trên 60 năm, thương hiệu hàng đầu phía Nam. 
Tỷ lệ có việc làm đúng chuyên ngành đào tạo rất cao, trên 90%. Được các tập đoàn, doanh nghiệp hàng đầu ưu tiên tuyển dụng.
Chương trình đào tạo đạt chuẩn quốc tế và khu vực, các chương trình liên kết quốc tế, đáp ứng xu thế hội nhập và đào tạo công dân toàn cầu.
Đội ngũ giảng viên được đào tạo bài bản, trình độ cao, nhiều kinh nghiệm thực tiễn, tận tâm. CBVC phục vụ chuyên nghiệp, chu đáo.
Phòng học, phòng Lab/thực tập tiên tiến, đầy đủ, đa dạng; 100% được trang bị máy lạnh.
Đầy ắp các sân chơi học thuật, câu lạc bộ nghiên cứu khoa học, sáng tạo - khởi nghiệp.
Các hoạt động văn thể mỹ đa dạng, hấp dẫn giúp sinh viên phát triển toàn diện.
Với triết lý giáo dục Nhân Bản, Trường dành quỹ học bổng lớn hỗ trợ sinh viên. Không để sinh viên phải bỏ học vì không có tiền đóng học phí.
Khuôn viên Trường trên 17 hecta - xanh - sạch - đẹp; là nơi lý tưởng

In [24]:
query = "HCMUTE có những ngành đào tạo nào trong năm 2025?"
print(generate_answer(query))

retrieved_context: 
:  10 lý do để bạn nên theo học tại HCMUTE: Trường Công lập với bề dày lịch sử trên 60 năm, thương hiệu hàng đầu phía Nam. 
Tỷ lệ có việc làm đúng chuyên ngành đào tạo rất cao, trên 90%. Được các tập đoàn, doanh nghiệp hàng đầu ưu tiên tuyển dụng.
Chương trình đào tạo đạt chuẩn quốc tế và khu vực, các chương trình liên kết quốc tế, đáp ứng xu thế hội nhập và đào tạo công dân toàn cầu.
Đội ngũ giảng viên được đào tạo bài bản, trình độ cao, nhiều kinh nghiệm thực tiễn, tận tâm. CBVC phục vụ chuyên nghiệp, chu đáo.
Phòng học, phòng Lab/thực tập tiên tiến, đầy đủ, đa dạng; 100% được trang bị máy lạnh.
Đầy ắp các sân chơi học thuật, câu lạc bộ nghiên cứu khoa học, sáng tạo - khởi nghiệp.
Các hoạt động văn thể mỹ đa dạng, hấp dẫn giúp sinh viên phát triển toàn diện.
Với triết lý giáo dục Nhân Bản, Trường dành quỹ học bổng lớn hỗ trợ sinh viên. Không để sinh viên phải bỏ học vì không có tiền đóng học phí.
Khuôn viên Trường trên 17 hecta - xanh - sạch - đẹp; là nơi lý tưởng

In [25]:
query = "Trường có những phương thức xét tuyển nào?"
print(generate_answer(query))

retrieved_context: 
:  Phương thức tuyển sinh: Theo 1 trong các  phương thức: 
- Xét tuyển dựa vào kết quả của kỳ thi Trung học phổ thông năm 2025 trên toàn quốc với các tổ hợp môn.
- Xét tuyển dựa vào tổng điểm trung bình học bạ 6 học kỳ hoặc tổng điểm trung bình học bạ 2 học kỳ của lớp 12 của 3 môn theo tổ hợp từ 18 điểm trở lên (không giới hạn ngưỡng điểm trung bình từng môn, áp dụng cho thí sinh tốt nghiệp năm 2024 và các năm trước). 
- Xét tuyển các điều kiện tương đương đối với thí sinh học chương trình giáo dục phổ thông của nước ngoài hoặc học ở nước ngoài.
- Xét tuyển theo điểm kỳ thi đánh giá năng lực.
- Xét tuyển sinh viên các trường Đại học.
Tổ hợp môn xét tuyển: Toán, Lý, Hóa (A00); Toán, Lý – Anh (A01); Toán, Văn, Anh (D01); Toán, Anh, Khoa học tự nhiên (D90).
Thí sinh đăng kí xét tuyển trực tuyến tại http://xettuyenqt.hcmute.edu.vn 

Xét tuyển theo lịch chung của Bộ GD&ĐT: Xét tuyển theo lịch chung của Bộ GD&ĐT
+ Phương thức 1: Xét tuyển theo kết quả điểm thi tốt nghiệp 

In [26]:
query = "Trường có ngành nào mới tuyển sinh trong năm 2025 không?"
print(generate_answer(query))

retrieved_context: 
:  Các ngành dự kiến mở mới năm 2025 học tại trường ĐH SPKT TP. HCM: ```markdown
| TT | Tên ngành đào tạo | Chương trình Đào tạo bằng tiếng Việt | Tổ hợp môn xét tuyển dự kiến (in đậm là môn chính nhân hệ số 2) | Mã ngành |
|----|--------------------|--------------------------------------|----------------------------------------------------------------------------------------|----------|
| 1  | Dinh dưỡng & Khoa học thực phẩm | 7720402V | (**Hóa** – Toán – Lý); (**Hóa** – Toán – Sinh); (**Hóa** – Toán – Anh); (**Hóa** – Toán – Công nghệ Công nghiệp). | 7720402V |
| 2  | Quản lý tài nguyên & môi trường (chuyên ngành Môi trường và Phát triển bền vững) | 7850101V | (**Toán** – Anh – Văn); (**Toán** – Anh – Hóa); (**Toán** – Anh – Sinh); (**Toán** – Anh – GDKT&PL). | 7850101V |
| 3  | Công nghệ tài chính | 7340205V | (**Toán** – Lý – Hóa); (**Toán** – Lý – Anh); (**Toán** – Văn – Anh); (**Toán** – Anh – Công nghệ Công nghiệp). | 7340205V |
| 4  | Quản trị kinh doanh | 7

In [27]:
query = "Có ngành nào đào tạo bằng tiếng Anh không?"
print(generate_answer(query))

retrieved_context: 
:  Ngành Sư phạm tiếng Anh (AUN-QA): Đào tạo chuyên sâu về giáo viên Tiếng Anh Kỹ thuật với khối kiến thức về kỹ thuật, năng lực sư phạm và các kỹ năng mềm cần thiết để dễ dàng thích nghi với mọi thay đổi trong môi trường giảng dạy tiếng Anh. Ngoài các môn đại cương và cơ sở về tiếng Anh, sinh viên còn được đào tạo một cách chuyên sâu và có hệ thống về phương pháp giảng dạy Tiếng Anh và các môn tiếng Anh chuyên ngành Công nghệ Thông tin, Thương mại, Công nghệ Môi trường, Điện-Điện tử, Cơ khí, Thiết kế Thời trang, Dinh dưỡng và Công nghệ Thực phẩm. 
Cơ hội việc làm: Giảng dạy tiếng Anh ở các cấp học giáo dục phổ thông, các trường nghề, trung học chuyên nghiệp, cao đẳng nghề, các trung tâm ngoại ngữ và các cơ sở đào tạo khác trong hệ thống giáo dục quốc dân. Ngoài ra cử nhân Sư phạm tiếng Anh còn có thể đảm nhận công việc trong các lĩnh vực khác như hướng dẫn viên du lịch, viết báo tiếng Anh.
Ngành Ngôn ngữ Anh: Đào tạo trang bị cho người học những kiến thức về khoa h

In [28]:
query = "Cho tôi biết về ngành Công nghệ thông tin của trường Đại học Sư phạm Kỹ thuật TP. Hồ Chí Minh?"
print(generate_answer(query))

retrieved_context: 
:  Ngành Vật lý kỹ thuật (Ngành mới tuyển sinh 2025): Ngành  Vật lý kỹ thuật của trường Đại học Sư phạm Kỹ thuật TP. Hồ Chí Minh là ngành định hướng công nghệ bán dẫn và cảm biến, đo lường, đào tạo mang tính liên ngành,  ứng dụng các nguyên lý vật lý và toán học để phân tích và giải quyết các vấn đề kỹ thuật và ứng dụng liên ngành. Mục tiêu tổng quát của ngành là đào tạo kỹ sư Vật lý Kỹ thuật có năng lực chuyên môn, được trang bị các kiến thức cơ sở vững vàng, có khả năng lãnh đạo, sáng tạo và khả năng tự học suốt đời trong lĩnh vực Vật lý kỹ thuật, đáp ứng nhu cầu lao động có trình độ kỹ thuật cao của đất nước. 
Cơ hội việc làm: Tốt nghiệp ngành Vật lý Kỹ thuật sinh viên có thể làm việc tại các doanh nghiệp hoạt động trong các lĩnh vực như: Các lĩnh vực liên quan đến vật liệu mới, vi điện tử – đo lường, lĩnh vực vật liệu điện tử, bán dẫn, vi mạch, vật liệu tổng hợp (composites), mực in thông minh,…
Phần giới thiệu: Trường Đại học Sư phạm Kỹ thuật TP. Hồ Chí Minh là

In [29]:
query = "Trường có ngành ngôn ngữ anh không?"
print(generate_answer(query))

retrieved_context: 
:  Ngành Sư phạm tiếng Anh (AUN-QA): Đào tạo chuyên sâu về giáo viên Tiếng Anh Kỹ thuật với khối kiến thức về kỹ thuật, năng lực sư phạm và các kỹ năng mềm cần thiết để dễ dàng thích nghi với mọi thay đổi trong môi trường giảng dạy tiếng Anh. Ngoài các môn đại cương và cơ sở về tiếng Anh, sinh viên còn được đào tạo một cách chuyên sâu và có hệ thống về phương pháp giảng dạy Tiếng Anh và các môn tiếng Anh chuyên ngành Công nghệ Thông tin, Thương mại, Công nghệ Môi trường, Điện-Điện tử, Cơ khí, Thiết kế Thời trang, Dinh dưỡng và Công nghệ Thực phẩm. 
Cơ hội việc làm: Giảng dạy tiếng Anh ở các cấp học giáo dục phổ thông, các trường nghề, trung học chuyên nghiệp, cao đẳng nghề, các trung tâm ngoại ngữ và các cơ sở đào tạo khác trong hệ thống giáo dục quốc dân. Ngoài ra cử nhân Sư phạm tiếng Anh còn có thể đảm nhận công việc trong các lĩnh vực khác như hướng dẫn viên du lịch, viết báo tiếng Anh.
Ngành Ngôn ngữ Anh: Đào tạo trang bị cho người học những kiến thức về khoa h

In [30]:
query = "Các ngành đào tạo năm 2025 học tại trường ĐH SPKT TP. HCM"
print(generate_answer(query))

retrieved_context: 
:  Các ngành dự kiến mở mới năm 2025 học tại trường ĐH SPKT TP. HCM: ```markdown
| TT | Tên ngành đào tạo | Chương trình Đào tạo bằng tiếng Việt | Tổ hợp môn xét tuyển dự kiến (in đậm là môn chính nhân hệ số 2) | Mã ngành |
|----|--------------------|--------------------------------------|----------------------------------------------------------------------------------------|----------|
| 1  | Dinh dưỡng & Khoa học thực phẩm | 7720402V | (**Hóa** – Toán – Lý); (**Hóa** – Toán – Sinh); (**Hóa** – Toán – Anh); (**Hóa** – Toán – Công nghệ Công nghiệp). | 7720402V |
| 2  | Quản lý tài nguyên & môi trường (chuyên ngành Môi trường và Phát triển bền vững) | 7850101V | (**Toán** – Anh – Văn); (**Toán** – Anh – Hóa); (**Toán** – Anh – Sinh); (**Toán** – Anh – GDKT&PL). | 7850101V |
| 3  | Công nghệ tài chính | 7340205V | (**Toán** – Lý – Hóa); (**Toán** – Lý – Anh); (**Toán** – Văn – Anh); (**Toán** – Anh – Công nghệ Công nghiệp). | 7340205V |
| 4  | Quản trị kinh doanh | 7

In [31]:
query = "Đối tượng tuyển sinh?"
print(generate_answer(query))

retrieved_context: 
:  Đối tượng tuyển sinh: Học sinh của tất cả các trường Trung học phổ thông (THPT) trên cả nước và học sinh học chương trình giáo dục phổ thông của nước ngoài hoặc học ở nước ngoài.

Đối tượng tuyển sinh: Học sinh của tất cả các trường Trung học phổ thông (THPT) trên cả nước.

Phương thức tuyển sinh: Theo 1 trong các  phương thức: 
- Xét tuyển dựa vào kết quả của kỳ thi Trung học phổ thông năm 2025 trên toàn quốc với các tổ hợp môn.
- Xét tuyển dựa vào tổng điểm trung bình học bạ 6 học kỳ hoặc tổng điểm trung bình học bạ 2 học kỳ của lớp 12 của 3 môn theo tổ hợp từ 18 điểm trở lên (không giới hạn ngưỡng điểm trung bình từng môn, áp dụng cho thí sinh tốt nghiệp năm 2024 và các năm trước). 
- Xét tuyển các điều kiện tương đương đối với thí sinh học chương trình giáo dục phổ thông của nước ngoài hoặc học ở nước ngoài.
- Xét tuyển theo điểm kỳ thi đánh giá năng lực.
- Xét tuyển sinh viên các trường Đại học.
Tổ hợp môn xét tuyển: Toán, Lý, Hóa (A00); 

In [ ]:
# query = "HCMUTE có những ngành đào tạo nào trong năm 2025?"
# print(generate_answer(query))

In [32]:
query = "Cho em hỏi em đạt giải 3 cuộc thi khoa học kỹ thuật cấp tỉnh có ưu tiên gì khi xét tuyển đại học ạ?"
print(generate_answer(query))

retrieved_context: 
:  Chính sách khuyến khích tài năng: Năm 2025 Trường dành 60 tỷ đồng để cấp học bổng cho sinh viên. 
+  Cấp học bổng khuyến tài cho thí sinh trúng tuyển có tổng điểm thi THPT 2025 (không tính điểm ưu tiên, điểm thưởng) của 3 môn xét tuyển từ 26 điểm trở lên; mỗi điểm thưởng 1.000.000đ; mỗi ngành chọn 1 thí sinh có điểm cao nhất.                                          
+  Cấp học bổng học kỳ đầu tiên, các học kỳ tiếp theo thì căn cứ vào kết quả học tập của học kỳ trước đó để cấp học bổng:
- Có giá trị bằng 50% học phí cho thí sinh nữ học các ngành kỹ thuật (*).
- Có giá trị bằng 20% học phí cho thí sinh có anh, chị em ruột đã hoặc đang học tại trường.
+ Ngành Sư phạm Anh và Ngành Sư phạm Công nghệ: Miễn học phí trong 4 năm học và còn được nhận tiền sinh hoạt phí 3,6 triệu đồng/tháng.
THÔNG TIN VỀ TUYỂN SINH HỆ ĐẠI HỌC CHÍNH QUY: Tuyển thẳng và ưu tiên xét tuyển: Nhận hồ sơ đăng ký từ ngày 01/4 – 30/5/2025 tại http://xettuyen.hcmute.edu.vn
+ Tuyển thẳng theo quy chế

In [33]:
query = "Dạ cho mình hỏi là năm nay trường mình có cho quy đổi điểm tiếng Anh không ạ?"
print(generate_answer(query))

retrieved_context: 
:  Quy đổi điểm tiếng Anh: Thí sinh có chứng chỉ IELTS và tương đương từ 4.5 trở lên có thể sử dụng để quy đổi điểm tiếng Anh để xét tuyển các tổ hợp có môn tiếng Anh thông qua hệ thống quy đổi của Nhà trường (áp dụng cho: ưu tiên xét tuyển, xét tuyển bằng học bạ THPT, điểm thi tốt nghiệp THPT 2024).

Bảng quy đổi chi tiết: ```markdown
| IELTS    | 4.5   | 5.0   | 5.5   | 6.0   | 6.5   | >= 7.0 |
|----------|-------|-------|-------|-------|-------|--------|
| Điểm tiếng Anh quy đổi cho các ngành | 7,5  | 8,0  | 8,5  | 9,0  | 9,5  | 10    |
VSTEP 


: Theo Thông tư số 23/2021/TT-BGDĐT ngày 30 tháng 8 năm 2021 của Bộ trưởng Bộ Giáo dục và Đào tạo.
```markdown
| Chứng chỉ/Văn bằng | Thang điểm | Điểm IELTS | Điểm quy đổi |
|---------------------|------------|------------|--------------|
| VSTEP               | B1         | 4.5        | 7,5          |
|                     | B2         | 6.0        | 9,0          |


\ --- 


Năm nay trường mình sẽ áp dụng hệ thống qu

In [34]:
query = "Dạ cho mình hỏi là năm nay trường mình có cho quy đổi điểm Ielts sang điểm thi tốt nghiệp Anh văn không ạ?"
print(generate_answer(query))

retrieved_context: 
:  Quy đổi điểm tiếng Anh: Thí sinh có chứng chỉ IELTS và tương đương từ 4.5 trở lên có thể sử dụng để quy đổi điểm tiếng Anh để xét tuyển các tổ hợp có môn tiếng Anh thông qua hệ thống quy đổi của Nhà trường (áp dụng cho: ưu tiên xét tuyển, xét tuyển bằng học bạ THPT, điểm thi tốt nghiệp THPT 2024).

Bảng quy đổi chi tiết: ```markdown
| IELTS    | 4.5   | 5.0   | 5.5   | 6.0   | 6.5   | >= 7.0 |
|----------|-------|-------|-------|-------|-------|--------|
| Điểm tiếng Anh quy đổi cho các ngành | 7,5  | 8,0  | 8,5  | 9,0  | 9,5  | 10    |
VSTEP 


: Theo Thông tư số 23/2021/TT-BGDĐT ngày 30 tháng 8 năm 2021 của Bộ trưởng Bộ Giáo dục và Đào tạo.
```markdown
| Chứng chỉ/Văn bằng | Thang điểm | Điểm IELTS | Điểm quy đổi |
|---------------------|------------|------------|--------------|
| VSTEP               | B1         | 4.5        | 7,5          |
|                     | B2         | 6.0        | 9,0          |


\ --- 


Đáp án là không. Theo thông tin trên, năm 

In [35]:
query = "Dạ cho em hỏi là mấy anh chị nghĩ là trường mình sẽ xét tổ hợp có môn công nghệ hoặc tin học không ạ"
print(generate_answer(query))

retrieved_context: 
:  Ghi chú: Ghi chú: 
- Khối xét tuyển cho tất cả các ngành, thí sinh chọn 1 trong 4 tổ hợp: Toán, Lý, Hóa (A00); Toán, Lý – Anh (A01); Toán, Văn, Anh (D01); Khoa học tự nhiên (D90).
Phương thức tuyển sinh: Theo 1 trong các  phương thức: 
- Xét tuyển dựa vào kết quả của kỳ thi Trung học phổ thông năm 2025 trên toàn quốc với các tổ hợp môn.
- Xét tuyển dựa vào tổng điểm trung bình học bạ 6 học kỳ hoặc tổng điểm trung bình học bạ 2 học kỳ của lớp 12 của 3 môn theo tổ hợp từ 18 điểm trở lên (không giới hạn ngưỡng điểm trung bình từng môn, áp dụng cho thí sinh tốt nghiệp năm 2024 và các năm trước). 
- Xét tuyển các điều kiện tương đương đối với thí sinh học chương trình giáo dục phổ thông của nước ngoài hoặc học ở nước ngoài.
- Xét tuyển theo điểm kỳ thi đánh giá năng lực.
- Xét tuyển sinh viên các trường Đại học.
Tổ hợp môn xét tuyển: Toán, Lý, Hóa (A00); Toán, Lý – Anh (A01); Toán, Văn, Anh (D01); Toán, Anh, Khoa học tự nhiên (D90).
Thí sinh đăng kí xét tuyển trực tuy

In [37]:
query = "Các đối tượng được ưu tiên khi xét tuyển đại học ạ?"
print(generate_answer(query))

retrieved_context: 
:  THÔNG TIN VỀ TUYỂN SINH HỆ ĐẠI HỌC CHÍNH QUY: Tuyển thẳng và ưu tiên xét tuyển: Nhận hồ sơ đăng ký từ ngày 01/4 – 30/5/2025 tại http://xettuyen.hcmute.edu.vn
+ Tuyển thẳng theo quy chế của Bộ GD&ĐT.
+ Ưu tiên xét tuyển theo Đề án tuyển sinh của trường (Xét tuyển thí sinh các Trường THPT có ký kết hợp tác)
* Trường Tổ chức thi các môn năng khiếu để xét tuyển vào 4 ngành: Thiết kế thời trang; Thiết kế đồ họa; Kiến trúc; Kiến trúc nội thất. Nhận hồ sơ đăng ký thi môn năng khiếu từ ngày 01/4 – 30/5/2025 tại http://xettuyen.hcmute.edu.vn
Để tăng khả năng trúng tuyển, thí sinh được khuyến nghị đăng ký nhiều phương thức và nhiều nguyện vọng (các nguyện vọng được xét theo thứ tự ưu tiên; nguyện vọng 1 là ưu tiên cao nhất).
Kinh nghiệm qua các năm: mỗi thí sinh đăng ký từ 5 - 7 nguyện vọng, trong đó nguyện vọng từ 1 - 3 nên chọn ĐH SPKT TP. HCM; mỗi mã ngành chỉ đăng ký một tổ hợp có điểm cao nhất. 
Thông tin chi tiết xem tại website: http://t

In [38]:
query = "Em muốn hỏi thông tin về các loại học bổng của trường ạ?"
print(generate_answer(query))

retrieved_context: 
:  Chính sách khuyến khích tài năng: Năm 2025 Trường dành 60 tỷ đồng để cấp học bổng cho sinh viên. 
+  Cấp học bổng khuyến tài cho thí sinh trúng tuyển có tổng điểm thi THPT 2025 (không tính điểm ưu tiên, điểm thưởng) của 3 môn xét tuyển từ 26 điểm trở lên; mỗi điểm thưởng 1.000.000đ; mỗi ngành chọn 1 thí sinh có điểm cao nhất.                                          
+  Cấp học bổng học kỳ đầu tiên, các học kỳ tiếp theo thì căn cứ vào kết quả học tập của học kỳ trước đó để cấp học bổng:
- Có giá trị bằng 50% học phí cho thí sinh nữ học các ngành kỹ thuật (*).
- Có giá trị bằng 20% học phí cho thí sinh có anh, chị em ruột đã hoặc đang học tại trường.
+ Ngành Sư phạm Anh và Ngành Sư phạm Công nghệ: Miễn học phí trong 4 năm học và còn được nhận tiền sinh hoạt phí 3,6 triệu đồng/tháng.
CHÍNH SÁCH KHUYẾN KHÍCH TÀI NĂNG: Học bổng khuyến khích theo quy định của Nhà trường;
Học bổng chuyển tiếp giai đoạn học ở nước ngoài: giảm từ 15% đến 100% học phí (tuỳ theo điều kiện 

In [39]:
query = "Hãy nêu cho em biết 1 vài lý do để em mạnh dạn chọn HCMUTE để học đại học?"
print(generate_answer(query))

retrieved_context: 
:  10 lý do để bạn nên theo học tại HCMUTE: Trường Công lập với bề dày lịch sử trên 60 năm, thương hiệu hàng đầu phía Nam. 
Tỷ lệ có việc làm đúng chuyên ngành đào tạo rất cao, trên 90%. Được các tập đoàn, doanh nghiệp hàng đầu ưu tiên tuyển dụng.
Chương trình đào tạo đạt chuẩn quốc tế và khu vực, các chương trình liên kết quốc tế, đáp ứng xu thế hội nhập và đào tạo công dân toàn cầu.
Đội ngũ giảng viên được đào tạo bài bản, trình độ cao, nhiều kinh nghiệm thực tiễn, tận tâm. CBVC phục vụ chuyên nghiệp, chu đáo.
Phòng học, phòng Lab/thực tập tiên tiến, đầy đủ, đa dạng; 100% được trang bị máy lạnh.
Đầy ắp các sân chơi học thuật, câu lạc bộ nghiên cứu khoa học, sáng tạo - khởi nghiệp.
Các hoạt động văn thể mỹ đa dạng, hấp dẫn giúp sinh viên phát triển toàn diện.
Với triết lý giáo dục Nhân Bản, Trường dành quỹ học bổng lớn hỗ trợ sinh viên. Không để sinh viên phải bỏ học vì không có tiền đóng học phí.
Khuôn viên Trường trên 17 hecta - xanh - sạch - đẹp; là nơi lý tưởng

In [40]:
query = "Chương trình đào tạo có chú trọng thực hành không hay chủ yếu là lý thuyết?"
print(generate_answer(query))

retrieved_context: 
:  Chương trình đào tạo Kỹ thuật Thiết kế vi mạch: Đào tạo chuyên sâu trong lĩnh vực kỹ thuật điện tử, thiết kế và phát triển các vi mạch điện tử, kiến thức về hệ thống tích hợp các linh kiện điện tử như transistor, điện trở, tụ điện, và các thành phần khác, trên một nền chất bán dẫn như silic...
Cơ hội việc làm: Đảm nhận các công việc trong lĩnh vực thiết kế và chế tạo vi mạch bản tại các công ty hoạt động trong lĩnh vực thiết kế và chế tạo vi mạch bán dẫn cũng như các lĩnh vực liên quan khác.
Ngành Kinh doanh quốc tế : Đào tạo các kiến thức chuyên sâu về hoạt động giao dịch kinh doanh được thực hiện giữa các quốc gia, nhằm thoả mãn các mục tiêu kinh doanh của doanh nghiệp, cá nhân và các tổ chức kinh tế, trang bị kiến thức tổng quan về quản trị kinh doanh xuất nhập khẩu, về chiến lược, chiến thuật kinh doanh xuyên quốc gia, các kiến thức về hoạt động thương mại, giao dịch được thực hiện giữa các quốc gia, nhằm thỏa mãn mục tiêu kinh doanh của doanh nghiệp, cá

In [44]:
query = "Các chương trình học bổng của trường đh sư phạm kỹ thuật?"
print(generate_answer(query))

retrieved_context: 
:  Chính sách khuyến khích tài năng: Năm 2025 Trường dành 60 tỷ đồng để cấp học bổng cho sinh viên. 
+  Cấp học bổng khuyến tài cho thí sinh trúng tuyển có tổng điểm thi THPT 2025 (không tính điểm ưu tiên, điểm thưởng) của 3 môn xét tuyển từ 26 điểm trở lên; mỗi điểm thưởng 1.000.000đ; mỗi ngành chọn 1 thí sinh có điểm cao nhất.                                          
+  Cấp học bổng học kỳ đầu tiên, các học kỳ tiếp theo thì căn cứ vào kết quả học tập của học kỳ trước đó để cấp học bổng:
- Có giá trị bằng 50% học phí cho thí sinh nữ học các ngành kỹ thuật (*).
- Có giá trị bằng 20% học phí cho thí sinh có anh, chị em ruột đã hoặc đang học tại trường.
+ Ngành Sư phạm Anh và Ngành Sư phạm Công nghệ: Miễn học phí trong 4 năm học và còn được nhận tiền sinh hoạt phí 3,6 triệu đồng/tháng.
Các ngành đào tạo năm 2025 học tại trường ĐH SPKT TP. HCM: ```markdown
| TT | Tên ngành đào tạo \n Cấp học bổng học kỳ 1 năm học đầu tiên: bằng 50% học phí cho nữ học 6 nga